# Wrap-up Web Scraping and Intro SQL with Python
v.ekc

We close out web scraping with a full table-parsing example, then pivot to SQL — using Python's `sqlite3` library to create, query, and exchange data between databases and pandas.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Web Scraping Wrap-up |
| 3 | Working with Databases — SQLite |
| 4 | Reading SQL Results into Pandas |
| 5 | Saving DataFrames to SQLite |
| 6 | 🔬 Activity — Chinook Database |
| Appendix | Quick Reference |

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

---
## 2. Web Scraping Wrap-up

Continuing from last class — extracting the ethnicity table from the Cal Poly Humboldt IRAR page and building a clean DataFrame.

In [ ]:
# Get the data
cph_stats = requests.get('https://www.humboldt.edu/irar/fall-semester-fast-facts')
cph_stats.status_code

In [ ]:
# Beautify the data
cph_soup = BeautifulSoup(cph_stats.text, 'html.parser')

In [ ]:
# Pretty!
print(cph_soup.prettify())

In [ ]:
# How many tables are there?
len(cph_soup.find_all('table'))

In [ ]:
# Tables are labeled with h3 headers
cph_soup.find_all('h3')

In [ ]:
# Let's just focus on one of the tables
student_ethnicity_table = cph_soup.find_all('table')[3]

In [ ]:
# Look at the rows
student_ethnicity_table.find_all('tr')

In [ ]:
# Look at a single row
student_ethnicity_table.find_all('tr')[0]

In [ ]:
# Look at all the data points in that row
student_ethnicity_table.find_all('tr')[0].find_all('td')

In [ ]:
# Look at one specific data point
student_ethnicity_table.find_all('tr')[0].find_all('td')[1]

In [ ]:
# Get the text of that data point 
student_ethnicity_table.find_all('tr')[0].find_all('td')[1].text

### Create a Pandas DataFrame

We'll start with the manual approach.

In [ ]:
# Create a nested list with the data
table_vals = []

for i in student_ethnicity_table.find_all('tr'):
    row_i = []
    for j in i.find_all('td'):
        row_i.append(j.text)
    table_vals.append(row_i)

In [ ]:
# Check out the result
table_vals

In [ ]:
# Make it a dataframe
df = pd.DataFrame(table_vals)
df

In [ ]:
# Clean it up (reset column labels)
df.columns = df.iloc[0]
df.drop(0,inplace=True)
df

In [ ]:
df.columns[0]

In [ ]:
# Clean it up (reset row labels)
df.set_index('\xa0',inplace=True)

In [ ]:
df

In [ ]:
# Would require further cleanup
df.dtypes

There is also a Pandas approach.

In [ ]:
# We've taken a chunk of html in a table that we want to parse
student_ethnicity_table

In [ ]:
# What is the type
type(student_ethnicity_table)

In [ ]:
# Read the html into a dataframe
pd.read_html(str(student_ethnicity_table))[0]

In [ ]:
# We can use it on the whole page if we want to
pd.read_html(cph_stats.text)

---
## 3. Working with Databases — SQLite

SQLite is a lightweight, file-based database that's great for learning SQL. Python's built-in `sqlite3` library gives you a connection and cursor to execute SQL directly.

### 📋 Board Reference

| Step | Code | Notes |
|------|------|-------|
| Connect | `sqlite3.connect('file.db')` | Creates the file if it doesn't exist |
| Cursor | `conn.cursor()` | Executes SQL statements |
| Execute | `cursor.execute("SQL")` | Run a single statement |
| Execute many | `cursor.executemany("SQL", list)` | Insert multiple rows |
| Commit | `conn.commit()` | Save changes |
| Close | `cursor.close(); conn.close()` | Always close when done |

In [ ]:
# When we want to work with datbases
import sqlite3

In [ ]:
# establish a connection with the database (this will create one if it doesn't exist)
dbconn = sqlite3.connect('demo.db')

In [ ]:
type(dbconn)

In [ ]:
# Check to see if your database is in your working directory
!ls

In [ ]:
# Create a cursor to use to execute SQL statements
cursor = dbconn.cursor()

In [ ]:
# Check data type
type(cursor)

### Use the cursor to execute SQL statements to the database.
Note:  We always execute a commit after the statement.

In [ ]:
# Create a table in our database
cursor.execute("""
CREATE TABLE  books
(title TEXT, author TEXT, price FLOAT, year INTEGER)
""")
dbconn.commit()

In [ ]:
# Add rows to our table
cursor.execute("""
    INSERT INTO books VALUES 
    ("The Great Gatsby", "F. Scott Fitzgerald", 5.15, 1925)
""")
cursor.execute("""
    INSERT INTO books VALUES 
    ("1984", "George Orwell", 7.24, 1961)
    """)

dbconn.commit()

In [ ]:
# Add multiple rows to our table
cursor.execute("""
    INSERT INTO books VALUES
        ("Pride and Prejudice", "Jane Austen", 12.99, 1813),
        ("To Kill a Mockingbird", "Harper Lee", 15.50, 1960)
""")
dbconn.commit()

In [ ]:
# another way to add multiple rows
books_list = [
    ("The Catcher in the Rye", "J.D. Salinger", 14.99, 1951),
    ("The Lord of the Rings", "J.R.R. Tolkien", 19.95, 1954),
    ("Moby-Dick", "Herman Melville", 11.75, 1851)
]

cursor.executemany("""
    INSERT INTO books VALUES (?,?,?,?)
""", books_list)
dbconn.commit()

In [ ]:
# Query data
lstbooks = cursor.execute('''SELECT * FROM books''').fetchall()
print(lstbooks)

In [ ]:
# Turn this into a dataframe
pd.DataFrame(lstbooks)

---
## 4. Reading SQL Results into Pandas

`pd.read_sql_query()` is cleaner than fetching and converting manually — it handles column names automatically.

In [ ]:
# convert query results to a dataframe
dfbook = pd.read_sql_query("SELECT * FROM books", dbconn)
dfbook

In [ ]:
dfbook.dtypes

In [ ]:
# query columns
pd.read_sql_query('''SELECT title,author,year FROM books''', dbconn)

In [ ]:
# Query rows based on condition
pd.read_sql_query('''SELECT * FROM books WHERE price < 10''', dbconn)

In [ ]:
# Query row based on position
pd.read_sql_query('''SELECT * FROM books WHERE ROWID = 3''', dbconn)

In [ ]:
# Query multiple rows based on position
pd.read_sql_query('''SELECT * FROM books LIMIT 3 OFFSET 2''', dbconn)

In [ ]:
# Query every other row
pd.read_sql_query('''SELECT * FROM (
               SELECT *, ROW_NUMBER() OVER () AS row_num FROM books
           ) AS numbered_rows
           WHERE row_num % 2 = 1''', dbconn)

---
## 5. Saving DataFrames to SQLite

`df.to_sql()` lets you write a pandas DataFrame directly into a database table — useful for persisting cleaned data.

In [ ]:
# Suppose we have another dataframe
another_df = pd.read_csv('earthquakes.csv')  
another_df

In [ ]:
# Add this dataframe to our database
#if_exists{'fail', 'replace', 'append'}, default 'fail'
another_df.to_sql('earthquakes', con=dbconn, if_exists='replace')

In [ ]:
# Check if it is in the database
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type = 'table'", dbconn)

In [ ]:
cursor.execute('''DROP TABLE books''')
dbconn.commit()

In [ ]:
# Check if it is in the database
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type = 'table'", dbconn)

In [ ]:
cursor.execute('''DROP TABLE earthquakes''')
dbconn.commit()

In [ ]:
# Check if it is in the database
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type = 'table'", dbconn)

In [ ]:
# Close your cursor and connection when you are done
cursor.close()
dbconn.close()

---
## 6. 🔬 Activity — Chinook Database

Take a look at SQLite's Sample Database: [Chinook](http://www.sqlitetutorial.net/sqlite-sample-database/) — a music store database with tables for tracks, albums, artists, customers, and invoices.

---
### Try It 1: Explore the Chinook Database 😊

1. Check to see if `chinook.db` is in your working directory.

2. Connect to the chinook database and create a cursor.

3. Check what tables are in the database.

4. Read the `tracks` table into a pandas DataFrame.

5. Read the `albums` table into a pandas DataFrame.

6. Combine your `tracks_df` and `album_df` (inner join).

7. Close your connection and cursor.

In [ ]:
# 1. Check working directory


In [ ]:
# 2. Connect and create cursor
conn = ...
cur = ...

In [ ]:
# 3. Check what tables are in the database


In [ ]:
# 4. Read tracks table
tracks_df = ...
tracks_df.head()

In [ ]:
# 5. Read albums table
album_df = ...
album_df.head()

In [ ]:
# 6. Combine tracks and albums (inner join)


In [ ]:
# 7. Close connection and cursor


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*The Chinook `tracks` and `albums` tables share an `AlbumId` key — a standard inner merge will pick it up automatically.*

```python
# 1.
!ls

# 2.
conn = sqlite3.connect('chinook.db')
cur = conn.cursor()

# 3.
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type='table'", conn)

# 4.
tracks_df = pd.read_sql_query('SELECT * FROM tracks', conn)

# 5.
album_df = pd.read_sql_query('SELECT * FROM albums', conn)

# 6.
tracks_df.merge(album_df)  # inner join on AlbumId

# 7.
cur.close()
conn.close()
```

</details>

---
## Appendix — SQLite + Pandas Quick Reference

### Connect and create
```python
import sqlite3
conn = sqlite3.connect('mydb.db')
cursor = conn.cursor()

cursor.execute("CREATE TABLE t (col1 TEXT, col2 INT)")
cursor.execute("INSERT INTO t VALUES ('a', 1)")
cursor.executemany("INSERT INTO t VALUES (?,?)", [('b',2), ('c',3)])
conn.commit()
```

### Query into pandas
```python
pd.read_sql_query("SELECT * FROM t", conn)
pd.read_sql_query("SELECT col1 FROM t WHERE col2 > 1", conn)
pd.read_sql_query("SELECT * FROM t LIMIT 5 OFFSET 2", conn)
```

### DataFrame → database
```python
df.to_sql('table_name', con=conn, if_exists='replace')
# if_exists: 'fail' | 'replace' | 'append'
```

### Cleanup
```python
cursor.execute("DROP TABLE t")
conn.commit()
cursor.close()
conn.close()
```